# Q. Bayesian Estimation of a User Ability Parameter from Item Responses

An online learning platform presents a user with a sequence of $n$ multiple-choice questions **one at a time**. Each question is either answered correctly or incorrectly, allowing the platform to update its estimate of the user's ability dynamically after every response.

Let $Y_i$ denote the user's response to the $i$-th item encountered:

$$Y_i=
\begin{cases}
1, & \text{if the user answers item } i \text{ correctly},\\
0, & \text{if the user answers item } i \text{ incorrectly}.
\end{cases}$$

The platform assumes that the probability of a correct response is governed by a two-parameter logistic (2PL) item response model. Specifically, conditional on the user's latent ability parameter $\Theta=\theta$, the response probability for item $i$ is:

$$P(Y_i=1\mid \Theta=\theta)=p_i(\theta)=\frac{1}{1+e^{-a_i(\theta-b_i)}},$$

where $a_i>0$ is the known discrimination parameter, and $b_i$ is the known difficulty parameter of item $i$.

Let $\mathbf{y}^{(k)} = (y_1, y_2, \dots, y_k)$ represent the **running vector of observed responses** up to the current step $k$ (where $1 \le k \le n$).

Before observing any responses, the platform initializes the user's latent ability estimate with a standard normal prior distribution:

$$f_{\Theta}^{(0)}(\theta) = \frac{1}{\sqrt{2\pi}} \exp\left(-\frac{\theta^2}{2}\right) \quad \text{implying} \quad \Theta \sim \mathscr{N}(0,1).$$

As the user progresses, the posterior distribution at step $k-1$ serves as the prior distribution for step $k$.

---

### Tasks

1. **Visualizing the Mechanics:** Plot $P(Y_i=1\mid \Theta=\theta)$ vs $\theta$ using Plotly for two distinct values of $a_i$, where one of those $a_i$ values is paired with three different difficulty values of $b_i$. Interpret how moving $b_i$ shifts the curve horizontally.
2. **Sequential Likelihood Contribution:** Write down the likelihood contribution $L(y_k \mid \theta)$ of a *single* new response $y_k$ at step $k$, given the latent ability $\theta$. Then, write down the joint likelihood function for the running history vector $\mathbf{y}^{(k)}$.
3. **Mathematical Formulation of the Running Update:** Write down the recursive relationship for the posterior density at step $k$, denoted $f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)})$, up to a proportionality constant, using the prior state $f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)})$ and the new observation $y_k$.
4. **Dynamic Shifting:** Explain how a correct answer ($y_k = 1$) to a highly difficult item (large $b_k$) mathematically shifts the peak of the running posterior density distribution relative to the previous step.
5. **Tracking Certainty and Sharpness:** Explain how the discrimination parameter $a_k$ of the current item alters the variance (or "sharpness") of the distribution during a running update. What happens when $a_k$ is very large versus very small?
6. **Numerical Implementation of a Running Grid:** Describe a algorithmic approach to numerically approximate and maintain this running posterior density function on a fixed grid of $\theta$-values. Explicitly state how you would perform the sequential normalization step computationally after an item is answered.


7. **Evaluating Convergence over the Timeline:** Suppose the user's true, hidden latent ability is $\theta_{\text{true}} = 0.75$. Write a Python script that extends your previous grid simulation to track the performance of the running estimators over a sequence of $n = 20$ items.
* **Simulate Responses:** Dynamically generate the user's responses $y_k \in \{0, 1\}$ at each step by comparing a random draw from a Uniform distribution $U(0,1)$ against the true response probability $p_k(\theta_{\text{true}})$. Give each item a random difficulty $b_k \sim \mathscr{N}(0, 1)$ and a random discrimination $a_k \sim \text{Uniform}(0.5, 2.0)$.
* **Track Estimators:** At each step $k$, calculate and store the running Posterior Mean ($\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$) and the running Maximum A Posteriori ($\widehat{\theta}_{\mathrm{MAP}}^{(k)}$) estimate.
* **Visualize:** Use Plotly to create a single line chart showing the progression of both estimators from step $0$ to $20$. Add a static horizontal reference line at $y = 0.75$ representing $\theta_{\text{true}}$.
* **Analysis:** Briefly explain how the distance between your estimators and $\theta_{\text{true}}$ changes as $k$ increases, and interpret what this implies about the platform's confidence in its measurement.


## Question 1: Bayesian Estimation of Latent Ability

### Theoretical Concepts

**1. Mechanics of the Difficulty Parameter**
Adjusting the difficulty parameter, $b_i$, translates the logistic probability curve horizontally along the ability axis ($\theta$). When $b_i$ increases, the curve moves to the right, indicating that a greater underlying ability is required for the user to achieve a 50% probability of answering correctly.

**2. Accumulating the Likelihood**
For a single new observation $y_k \in \{0, 1\}$, the likelihood follows a standard Bernoulli distribution:
$$L(y_k \mid \theta) = p_k(\theta)^{y_k} (1 - p_k(\theta))^{1 - y_k}$$
Assuming each response is independent given $\theta$, the joint likelihood for the entire sequence of answers $\mathbf{y}^{(k)}$ is the product of the individual step likelihoods:
$$L(\mathbf{y}^{(k)} \mid \theta) = \prod_{j=1}^{k} p_j(\theta)^{y_j} (1 - p_j(\theta))^{1 - y_j}$$

**3. Sequential Bayesian Updating**
Applying Bayes' theorem step-by-step, the updated posterior distribution at iteration $k$ is directly proportional to the previous step's posterior (which acts as the new prior) multiplied by the latest observation's likelihood:
$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)}) \times \left[ p_k(\theta)^{y_k} (1 - p_k(\theta))^{1 - y_k} \right]$$

**4. Evidence-Driven Shifts**
When a user successfully answers a very hard question ($y_k = 1$, high $b_k$), the resulting likelihood heavily weights the upper end of the $\theta$ scale. Multiplying this likelihood by the prior pushes the peak of the updated posterior sharply to the right, capturing the newly demonstrated proficiency.

**5. Discrimination and Distribution Sharpness**
The $a_k$ parameter dictates the steepness of the logistic function.
* **High $a_k$:** The probability transitions abruptly at the difficulty threshold. This yields strong, decisive evidence that rapidly shrinks the variance of the posterior, producing a tightly peaked distribution.
* **Low $a_k$:** The curve is gradual and provides ambiguous evidence. The posterior variance is only marginally reduced.

**6. Grid-Based Numerical Integration**
To approximate this process computationally without closed-form algebra:
* Establish a fixed, dense array of $\theta$ values (e.g., spanning -4 to 4).
* Set up an initial prior probability array, such as a standard normal distribution, over this grid.
* For each new response $y_k$, generate a likelihood array evaluated across the identical grid.
* Multiply the prior and likelihood arrays element-wise to form the unnormalized posterior.
* **Normalization:** Use numerical integration (like the trapezoidal rule) to find the total area under the unnormalized curve. Divide the array by this scalar area so the final probabilities sum to exactly 1.


Implementation and Simulation (Tasks 1 & 7)

In [2]:
import numpy as np
import plotly.graph_objects as go
from scipy.stats import norm

# --- Task 1: Visualizing the Mechanics ---
theta_grid = np.linspace(-4, 4, 1000)

def p_theta(theta, a, b):
    return 1 / (1 + np.exp(-a * (theta - b)))

fig1 = go.Figure()
# Fixed 'a', varying 'b'
for b in [-1, 0, 1]:
    fig1.add_trace(go.Scatter(x=theta_grid, y=p_theta(theta_grid, a=1.5, b=b),
                              name=f"a=1.5, b={b}"))
# Different 'a' for contrast
fig1.add_trace(go.Scatter(x=theta_grid, y=p_theta(theta_grid, a=0.5, b=0),
                          name="a=0.5, b=0", line=dict(dash='dash')))
fig1.update_layout(title="2PL IRT Model Probabilities", xaxis_title="Theta (Ability)", yaxis_title="P(Y=1 | Theta)")
fig1.show()

# --- Task 7: Evaluating Convergence over the Timeline ---
np.random.seed(42)
theta_true = 0.75
n_items = 20

# Simulate items
b_items = np.random.normal(0, 1, n_items)
a_items = np.random.uniform(0.5, 2.0, n_items)

# Initialize tracking arrays
theta_bayes = np.zeros(n_items + 1)
theta_map = np.zeros(n_items + 1)

# Initial Prior N(0,1)
posterior = norm.pdf(theta_grid, 0, 1)
theta_bayes[0] = np.trapezoid(theta_grid * posterior, theta_grid)
theta_map[0] = theta_grid[np.argmax(posterior)]

# Simulate and Update
for k in range(n_items):
    # Simulate response
    p_true = p_theta(theta_true, a_items[k], b_items[k])
    y_k = 1 if np.random.uniform(0, 1) < p_true else 0

    # Likelihood array
    likelihood = p_theta(theta_grid, a_items[k], b_items[k]) if y_k == 1 else (1 - p_theta(theta_grid, a_items[k], b_items[k]))

    # Update and Normalize via Trapezoidal Rule
    unnormalized_posterior = posterior * likelihood
    marginal_likelihood = np.trapezoid(unnormalized_posterior, theta_grid)
    posterior = unnormalized_posterior / marginal_likelihood

    # Store Estimators
    theta_bayes[k+1] = np.trapezoid(theta_grid * posterior, theta_grid)
    theta_map[k+1] = theta_grid[np.argmax(posterior)]

# Visualize estimators
fig2 = go.Figure()
steps = np.arange(n_items + 1)
fig2.add_trace(go.Scatter(x=steps, y=theta_bayes, mode='lines+markers', name="Posterior Mean"))
fig2.add_trace(go.Scatter(x=steps, y=theta_map, mode='lines+markers', name="MAP Estimate"))
fig2.add_hline(y=theta_true, line_dash="dash", line_color="red", annotation_text="True Theta")
fig2.update_layout(title="Running Bayesian Estimators vs True Latent Ability",
                   xaxis_title="Item Step (k)", yaxis_title="Estimated Theta")
fig2.show()

# Q. Bayesian Tracking of Click-Through Rates (CTR) via Conjugate Beta-Binomial Updates

An e-commerce platform wants to optimize its recommendation engine by dynamically estimating the click-through rate (CTR) of a newly launched advertisement. Since user traffic arrives continuously, the platform updates its belief about the advertisement's performance **one impression at a time** rather than waiting for large batch updates.

Let $\Theta = \theta$ represent the true, hidden conversion rate (probability of a click) of the advertisement, where $\theta \in [0, 1]$.

Let $Y_k$ denote a single user's interaction with the advertisement at time step $k$:

$$Y_k =
\begin{cases}
1, & \text{if the user clicks the advertisement}, \\
0, & \text{if the user does not click the advertisement}.
\end{cases}$$

The platform assumes that conditional on the true conversion rate $\Theta = \theta$, each user interaction is an independent Bernoulli trial:

$$P(Y_k = 1 \mid \Theta = \theta) = \theta$$

Let $\mathbf{y}^{(k)} = (y_1, y_2, \dots, y_k)$ represent the **running vector of observed user interactions** up to the current impression step $k$ (where $1 \le k \le n$).

Before observing any data, the platform assigns a flexible **Beta distribution** as the initial prior over the unknown parameter $\Theta$:

$$f_{\Theta}^{(0)}(\theta) = \frac{1}{\mathrm{B}(\alpha_0, \beta_0)} \theta^{\alpha_0 - 1} (1 - \theta)^{\beta_0 - 1} \quad \text{implying} \quad \Theta \sim \text{Beta}(\alpha_0, \beta_0)$$

where $\mathrm{B}(\cdot, \cdot)$ is the Beta function acting as the normalizing constant. Under a sequential framework, the posterior distribution at step $k-1$ serves directly as the prior distribution for step $k$.

---

**Tasks**

**1. Structural Probability and Properties**
Plot the probability density function (PDF) of a $\text{Beta}(\alpha, \beta)$ distribution using Plotly for three distinct parameter pairs:

* Uninformative state: $(\alpha=1, \beta=1)$
* Right-skewed state: $(\alpha=2, \beta=8)$
* Left-skewed state: $(\alpha=8, \beta=2)$

Interpret how changing the balance between $\alpha$ and $\beta$ shifts the center of mass of the density function over the domain $[0, 1]$.

**2. Sequential Likelihood and Joint History**

Write down the mathematical likelihood contribution $L(y_k \mid \theta)$ of a *single* isolated response $y_k$ at step $k$, given the click probability $\theta$. Following this, express the joint likelihood function for the running history vector $\mathbf{y}^{(k)}$.

**3. Closed-Form Analytical Updates (Conjugacy)**

Using Bayes' Theorem, derive the recursive algebraic relationship for the posterior density at step $k$, denoted as $f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)})$. Prove analytically that the posterior remains in the Beta family (**Beta-Binomial Conjugacy**) by explicitly writing down the closed-form update parameters $\alpha_k$ and $\beta_k$ as simple arithmetic updates of $\alpha_{k-1}$, $\beta_{k-1}$, and $y_k$. Also compute the **Posterior Mean** of the latent parameter $\Theta$ at time step $k$ (i.e. $\mathbb{E}[\Theta \mid \mathbf{Y}^{(k)}=\mathbf{y}^{(k)}]$).


**4. Dynamic Shifting Mechanics**

Explain how an observed click ($y_k = 1$) vs. a non-click ($y_k = 0$) shifts the peak of the running density distribution mathematically. Contrast this analytical framework against non-conjugate setups (such as the 2PL IRT model) where numerical grid integration is strictly required.

**5. Running Point Estimators**

State the exact closed-form equations used to evaluate the following point estimates at step $k$ directly from the updated shape parameters $\alpha_k$ and $\beta_k$:

* **Running Posterior Mean** ($\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$)
* **Running Maximum A Posteriori** ($\widehat{\theta}_{\mathrm{MAP}}^{(k)}$)

**6. Performance Tracking and Convergence Analysis**

Suppose the advertisement's true, hidden click-through rate is $\theta_{\text{true}} = 0.35$. Write a Python script to track the performance of your closed-form sequential estimators over a timeline of $n = 100$ impressions:

* **Initialize State:** Set the base prior parameters to $\alpha_0 = 1, \beta_0 = 1$ (representing uniform initial uncertainty).
* **Simulate Responses:** Dynamically generate user responses $y_k \in \{0, 1\}$ at each step by comparing a random draw from a Uniform distribution $U(0,1)$ against $\theta_{\text{true}}$.
* **Track Estimators:** Loop through each step, update $\alpha_k$ and $\beta_k$ analytically, and store the computed values for $\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$ and $\widehat{\theta}_{\mathrm{MAP}}^{(k)}$.
* **Visualize:** Use Plotly to create a single line chart showing the progression of both estimators from step $0$ to $100$. Add a static horizontal reference line at $y = 0.35$ representing $\theta_{\text{true}}$.
* **Analysis:** Explain how the distance between your estimators and $\theta_{\text{true}}$ responds as the sampling size $k$ approaches $100$. What does this imply about the accumulation of evidence over time relative to the choice of the initial prior?

## Question 2: Tracking Click-Through Rates (CTR) with Conjugate Priors

### Theoretical Concepts

**1. Shaping the Prior**
The relative sizes of $\alpha$ and $\beta$ determine where the Beta distribution's mass is concentrated on the $[0, 1]$ interval. Equal values center it precisely at 0.5. If $\alpha > \beta$, the mass leans to the right (implying a higher initial CTR assumption); if $\beta > \alpha$, it leans to the left.

**2. Likelihood of the Click Stream**
The likelihood for one binary event is:
$$L(y_k \mid \theta) = \theta^{y_k} (1 - \theta)^{1 - y_k}$$
Over a sequence of $k$ events, the joint likelihood aggregates the successes and failures:
$$L(\mathbf{y}^{(k)} \mid \theta) = \theta^{\sum_{i=1}^k y_i} (1 - \theta)^{k - \sum_{i=1}^k y_i}$$

**3. The Conjugacy Advantage**
Because the Beta prior and Binomial likelihood are conjugate pairs, updating the model via Bayes' theorem cleanly yields another Beta distribution:
$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto \theta^{(\alpha_{k-1} + y_k) - 1} (1-\theta)^{(\beta_{k-1} + 1 - y_k) - 1}$$
This allows us to bypass complex calculus in favor of exact arithmetic updates:
$$\alpha_k = \alpha_{k-1} + y_k$$
$$\beta_k = \beta_{k-1} + (1 - y_k)$$
The expected value (Posterior Mean) at step $k$ simplifies effortlessly to:
$$\mathbb{E}[\Theta \mid \mathbf{Y}^{(k)}=\mathbf{y}^{(k)}] = \frac{\alpha_k}{\alpha_k + \beta_k}$$

**4. Behavioral Updating**
Every click ($y_k=1$) adds 1 to $\alpha_k$, nudging the distribution rightward. Every skipped ad ($y_k=0$) adds 1 to $\beta_k$, nudging it leftward. Unlike the IRT model, which mandates numerical grids, this conjugate relationship allows for instantaneous structural updates to the probability distribution.

**5. Real-Time Estimators**
* **Running Posterior Mean:** $\widehat{\theta}_{\mathrm{Bayes}}^{(k)} = \frac{\alpha_k}{\alpha_k + \beta_k}$
* **Maximum A Posteriori (MAP):** $\widehat{\theta}_{\mathrm{MAP}}^{(k)} = \frac{\alpha_k - 1}{\alpha_k + \beta_k - 2}$ (applicable only when $\alpha_k, \beta_k > 1$)

Implementation and Simulation (Tasks 1 & 6)

In [3]:
import numpy as np
import plotly.graph_objects as go
from scipy.stats import beta

# --- Task 1: Beta Distribution Plotting ---
theta_domain = np.linspace(0, 1, 500)
fig3 = go.Figure()
params = [(1, 1, "Uninformative"), (2, 8, "Right-skewed"), (8, 2, "Left-skewed")]
for a, b, name in params:
    fig3.add_trace(go.Scatter(x=theta_domain, y=beta.pdf(theta_domain, a, b), name=f"Beta({a},{b}) - {name}"))
fig3.update_layout(title="Beta Distribution Shapes", xaxis_title="Theta", yaxis_title="Density")
fig3.show()

# --- Task 6: Tracking Estimators ---
np.random.seed(101)
theta_true = 0.35
n_impressions = 100

alpha_k, beta_k = 1, 1
mean_estimates = [alpha_k / (alpha_k + beta_k)]
map_estimates = [np.nan] # MAP undefined for alpha=1, beta=1

for k in range(n_impressions):
    # Simulate user interaction
    y_k = 1 if np.random.uniform(0, 1) < theta_true else 0

    # Analytical Conjugate Update
    alpha_k += y_k
    beta_k += (1 - y_k)

    # Track estimates
    mean_estimates.append(alpha_k / (alpha_k + beta_k))
    # MAP estimate
    if alpha_k > 1 or beta_k > 1:
        map_estimates.append((alpha_k - 1) / (alpha_k + beta_k - 2))
    else:
        map_estimates.append(np.nan)

fig4 = go.Figure()
steps = np.arange(n_impressions + 1)
fig4.add_trace(go.Scatter(x=steps, y=mean_estimates, mode='lines', name="Posterior Mean"))
fig4.add_trace(go.Scatter(x=steps, y=map_estimates, mode='lines', name="MAP Estimate"))
fig4.add_hline(y=theta_true, line_dash="dash", line_color="red", annotation_text="True CTR")
fig4.update_layout(title="Running CTR Estimators over Time", xaxis_title="Impression Step (k)", yaxis_title="Estimated CTR")
fig4.show()

# Q Bayesian Estimations for Structural Health Monitoring via Bounded Grid Updates

In aerospace and civil engineering, Structural Health Monitoring (SHM) is critical for detecting damage before a catastrophic failure occurs. Consider an aircraft wing or a bridge girder equipped with specialized vibration sensors. Over time, environmental fatigue or dynamic impacts can cause micro-fractures, resulting in a reduction of the component's mechanical stiffness.

Let $\Theta = \theta$ represent the structural **remaining stiffness efficiency factor**, where $\theta$ is physically bounded to the interval:

$$\theta \in (0, 1]$$

* $\theta = 1.0$ indicates a perfectly pristine, undamaged structural component.
* $\theta \to 0$ signifies critical degradation or severe structural cracking.

Let $K_{\text{nominal}}$ be the known, baseline stiffness of the structural component when it is entirely healthy. At each sequential inspection time step $k$ (where $k = 1, 2, \dots, n$), a sensor collects a noisy experimental stiffness measurement $y_k$.

Engineers model the degradation physics via a non-linear relationship with multiplicative log-normal measurement noise to prevent non-physical negative values:

$$y_k = \theta \cdot K_{\text{nominal}} \cdot e^{\epsilon_k}, \qquad \epsilon_k \sim \mathscr{N}(0, \sigma^2)$$

where $\sigma$ is the standard deviation of the sensor noise in log-space.

Let $\mathbf{y}^{(k)} = (y_1, y_2, \dots, y_k)$ represent the **running history vector of observed sensor readings** up to the current inspection milestone. Before deploying the sensors, engineers utilize an initial prior distribution $f_{\Theta}^{(0)}(\theta)$ over the domain $(0, 1]$ based on historical manufacturing specifications. As the sensor stream arrives, the posterior distribution calculated at step $k-1$ serves directly as the prior distribution for step $k$.

---

### **Tasks**

#### **1. Prior Belief Boundaries**

Before data collection begins, engineers assume the component is highly likely to be healthy, modeling this using a bounded Beta distribution as the initial prior: $\Theta \sim \text{Beta}(8, 1.5)$.

* Plot this initial prior density function using Plotly over the restricted physical domain $\theta \in [0.01, 1.0]$.
* Calculate the expected prior stiffness efficiency $\mathbb{E}[\Theta^{(0)}]$ analytically. Explain why this specific distribution serves as an appropriate initial prior for an engineering component assumed to be healthy.

#### **2. Structural Likelihood Formulation**

Using the change of variables or properties of the log-normal distribution, write down the mathematical likelihood contribution $L(y_k \mid \theta)$ of a *single* continuous sensor measurement $y_k$ at inspection step $k$, given the true stiffness factor $\theta$. Following this, write down the joint likelihood function for the running history vector $\mathbf{y}^{(k)}$.

#### **3. Mathematical Formulation of the Non-Conjugate Grid Update**

Explain why an exact closed-form analytical solution for the posterior density $f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)})$ does not exist when combining a Beta prior with this log-normal structural likelihood. Write down the recursive relationship for the posterior density at step $k$ up to a proportionality constant.

#### **4. Running Point Estimates**

Because a closed-form formula is unavailable, we must define point estimators through numerical integration. Write down the definite integral equations over the bounded domain $(0, 1]$ required to compute:

* The **Running Posterior Mean** ($\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$)
* The **Running Maximum A Posteriori** ($\widehat{\theta}_{\mathrm{MAP}}^{(k)}$)

#### **5. Algorithmic Grid Approximation and Normalization**

Describe the step-by-step numerical procedure to maintain this distribution on a discrete grid of $\theta$-values. Explicitly state how you would handle the boundary limits computationally and how you would perform the sequential normalization step using the trapezoidal rule after a new sensor reading $y_k$ is observed.

#### **6. Performance Tracking and Degradation Convergence Analysis**

Suppose an impact occurs, and the true, hidden remaining stiffness drops to $\theta_{\text{true}} = 0.68$. Write a Python script using Plotly to simulate an engineered monitoring timeline across $n = 15$ continuous sensor measurements ($K_{\text{nominal}} = 50.0 \text{ kN/mm}$, $\sigma = 0.15$):

* **Simulate Sensor Stream:** Programmatically generate noisy sensor readings $y_k$ by drawing random values from the underlying log-normal physics model centered at $\theta_{\text{true}}$.
* **Track Estimators:** Loop sequentially through each step. At each step, update the unnormalized grid, normalize it via `np.trapezoid`, and compute both $\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$ and $\widehat{\theta}_{\mathrm{MAP}}^{(k)}$.
* **Visualize Curves & Timeline:** Generate two plots:
1. A plot showing the progression of the full posterior density curves at milestones $k \in \{0, 1, 2, 5, 10, 15\}$.
2. A line chart tracking the convergence of both $\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$ and $\widehat{\theta}_{\mathrm{MAP}}^{(k)}$ from step $0$ to $15$ against a horizontal reference line at $\theta_{\text{true}} = 0.68$.


* **Analysis:** Evaluate the behavior of the distribution. How many sensor readings did it take for the system to overcome the initially optimistic "healthy" prior and confidently isolate the 68% damage state? What does the narrowing of the density curves imply about structural safety thresholds?

## Question 3: Structural Health Monitoring

### Theoretical Concepts

**1. Setting the Initial Belief**
The starting prior is modeled as $\Theta \sim \text{Beta}(8, 1.5)$, constrained to the domain $[0.01, 1.0]$. The expected initial stiffness is roughly $0.842$ ($\frac{8}{9.5}$). This distribution makes physical sense: it assumes a newly manufactured component is mostly intact (clustered near 1.0) while leaving statistical room for minor manufacturing flaws, rather than assuming absolute perfection.

**2. Noise-Injected Likelihood**
The sensor reading incorporates multiplicative log-normal noise, represented as $y_k = \theta \cdot K_{\text{nominal}} \cdot e^{\epsilon_k}$ where $\epsilon_k \sim \mathscr{N}(0, \sigma^2)$. Thus, the likelihood function is:
$$L(y_k \mid \theta) = \frac{1}{y_k \sigma \sqrt{2\pi}} \exp\left( - \frac{(\ln y_k - \ln(\theta K_{\text{nominal}}))^2}{2\sigma^2} \right)$$
The cumulative likelihood across multiple inspections is simply the product of these independent distributions.

**3. Non-Conjugate Processing**
Multiplying a bounded Beta prior by a Log-normal likelihood does not result in a standard, parameterized probability distribution. Because this pairing is non-conjugate, we must evaluate the posterior recursively through numerical means:
$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)}) \times \exp\left( - \frac{(\ln y_k - \ln(\theta K_{\text{nominal}}))^2}{2\sigma^2} \right)$$

**4. Extracting the Estimates**
Without a clean algebraic formula, we derive point estimates by analyzing the numerical density over the bounded domain $(0, 1]$:
* **Posterior Mean:** Computed via definite integration over the grid: $\widehat{\theta}_{\mathrm{Bayes}}^{(k)} = \int_{0.01}^{1.0} \theta \cdot f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \, d\theta$
* **MAP Estimate:** Found by locating the maximum density value directly on the numerical grid: $\widehat{\theta}_{\mathrm{MAP}}^{(k)} = \operatorname{arg\,max}_{\theta \in (0.01, 1.0]} f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)})$

**5. Grid Implementation Strategy**
* Discretize the $[0.01, 1.0]$ boundary into an array of $N$ points.
* Evaluate the Beta(8, 1.5) PDF to generate the prior array.
* For every incoming sensor reading $y_k$, compute the log-normal likelihood across the grid.
* Multiply the likelihood and prior arrays together element-wise.
* Normalize by integrating the resulting curve and dividing the array by that area to maintain a valid probability distribution.

In [4]:
import numpy as np
import plotly.graph_objects as go
from scipy.stats import beta, lognorm

# Parameters
K_nom = 50.0
sigma = 0.15
theta_true = 0.68
n_steps = 15

# Grid setup (Boundary handling)
theta_grid = np.linspace(0.01, 1.0, 1000)
prior = beta.pdf(theta_grid, 8, 1.5)

# --- Task 1: Plot Initial Prior ---
fig5 = go.Figure()
fig5.add_trace(go.Scatter(x=theta_grid, y=prior, name="Prior Beta(8, 1.5)"))
fig5.update_layout(title="Initial Structural Health Prior", xaxis_title="Theta (Stiffness)", yaxis_title="Density")
fig5.show()

# --- Task 6: Simulation ---
np.random.seed(42)
posteriors = {0: prior.copy()}
mean_est = [np.trapezoid(theta_grid * prior, theta_grid)]
map_est = [theta_grid[np.argmax(prior)]]

posterior = prior.copy()

for k in range(1, n_steps + 1):
    # Simulate log-normal sensor reading
    y_k = theta_true * K_nom * np.exp(np.random.normal(0, sigma))

    # Compute likelihood
    likelihood = (1 / (y_k * sigma * np.sqrt(2 * np.pi))) * np.exp(- (np.log(y_k) - np.log(theta_grid * K_nom))**2 / (2 * sigma**2))

    # Update and Normalize
    unnormalized = posterior * likelihood
    posterior = unnormalized / np.trapezoid(unnormalized, theta_grid)

    if k in [1, 2, 5, 10, 15]:
        posteriors[k] = posterior.copy()

    mean_est.append(np.trapezoid(theta_grid * posterior, theta_grid))
    map_est.append(theta_grid[np.argmax(posterior)])

# Visualize Posteriors
fig6 = go.Figure()
for k, dist in posteriors.items():
    fig6.add_trace(go.Scatter(x=theta_grid, y=dist, name=f"Step {k}"))
fig6.add_vline(x=theta_true, line_dash="dash", line_color="black", annotation_text="True Damage")
fig6.update_layout(title="Posterior Evolution over Sensor Readings", xaxis_title="Theta", yaxis_title="Density")
fig6.show()

# Visualize Convergence
fig7 = go.Figure()
steps = np.arange(n_steps + 1)
fig7.add_trace(go.Scatter(x=steps, y=mean_est, mode='lines+markers', name="Posterior Mean"))
fig7.add_trace(go.Scatter(x=steps, y=map_est, mode='lines+markers', name="MAP Estimate"))
fig7.add_hline(y=theta_true, line_dash="dash", line_color="red", annotation_text="True Theta")
fig7.update_layout(title="Estimator Convergence", xaxis_title="Inspection Step (k)", yaxis_title="Estimated Stiffness")
fig7.show()

Analysis: It takes roughly 3 to 5 sensor readings for the system to definitively overcome the initially optimistic prior and lock onto the 68% true damage state. The severe narrowing of the density curves (peaking heavily) implies that the system's variance is shrinking, allowing engineers to confidently establish rigid structural safety thresholds based on highly reliable degradation measurements.

# Q. Gaussian Mixture Clustering as Conditional Updating

Consider a dataset
$$
x_1,x_2,\dots,x_n\in\mathbb R^d.
$$
We wish to cluster these observations into $K$ groups. Instead of assigning each point deterministically to a cluster at the beginning, we introduce a latent random variable
$$
C_i\in{1,\dots,K},
$$
where $C_i=k$ means that the observation $x_i$ belongs to cluster $k$.
Let the prior probability of cluster membership be
$$
P(C_i=k)=\phi_k,
$$
where
$$
\phi_k\ge 0,
\qquad
\sum_{k=1}^K \phi_k=1.
$$

Conditional on $C_i=k$, assume that the observation $X_i$ is generated from a multivariate Gaussian distribution:
$$
X_i\mid C_i=k
\sim
\mathscr N(\mu_k,\Sigma_k),
$$
where
$$
\mu_k\in\mathbb R^d,
\qquad
\Sigma_k\in\mathbb R^{d\times d}
$$
are the mean vector and covariance matrix of cluster $k$.

The model parameters
$$
\phi_k,\mu_k,\Sigma_k,
\qquad k=1,\dots,K,
$$
are assumed to be fixed but unknown.

---

1. Deriving the Marginal Density:
Using the law of total probability, show that the marginal density of $X_i$ is
$$
p(x_i)=\sum_{k=1}^K
\phi_k
\mathscr N(x_i\mid \mu_k,\Sigma_k).
$$
Explain why this density is called a Gaussian mixture density.

---

2. Deriving the Posterior Cluster Probability:
For a fixed observation $x_i$, use Bayes' rule to derive
$$
P(C_i=k\mid X_i=x_i)=\frac{
P(X_i=x_i\mid C_i=k)P(C_i=k)
}{
\sum_{j=1}^K P(X_i=x_i\mid C_i=j)P(C_i=j)
}.
$$
Then substitute the Gaussian model and the cluster prior to obtain
$$
P(C_i=k\mid X_i=x_i)=\frac{
\phi_k\mathscr N(x_i\mid \mu_k,\Sigma_k)
}{
\sum_{j=1}^K
\phi_j\mathscr N(x_i\mid \mu_j,\Sigma_j)
}.
$$
This quantity is called the responsibility of cluster $k$ for data point $x_i$, and is denoted by
$$
\gamma_{ik}=P(C_i=k\mid X_i=x_i).
$$
Explain why $\gamma_{ik}$ may be interpreted as a posterior probability of cluster membership.

---

3. One-Hot Encoding of the Latent Cluster Variable:
Now define a one-hot encoded latent random vector
$$
Z_i=
\begin{bmatrix}
Z_{i1}\\
Z_{i2}\\
\vdots\\
Z_{iK}
\end{bmatrix},
$$
where
$$
Z_{ik}=\begin{cases}
1, & \text{if } C_i=k,\\
0, & \text{otherwise}.
\end{cases}
$$
Show that
$$
\mathbb E[Z_{ik}\mid X_i=x_i]=P(C_i=k\mid X_i=x_i).
$$
Hence show that
$$
\mathbb E[Z_i\mid X_i=x_i]=\begin{bmatrix}
\gamma_{i1}\\
\gamma_{i2}\\
\vdots\\
\gamma_{iK}
\end{bmatrix}.
$$
Conclude that the soft cluster assignment in a Gaussian mixture model is precisely the conditional expectation
$$
\mathbb E[Z_i\mid X_i=x_i].
$$

---

4. From Soft Assignment to Hard Clustering:
The vector
$$
\mathbb E[Z_i\mid X_i=x_i]
$$
gives a soft assignment of $x_i$ to all clusters. A hard cluster assignment can be obtained by choosing the cluster with the largest posterior probability:
$$
\widehat C_i=\operatorname{arg\,max}_{1\le k\le K}
\gamma_{ik}.
$$
Explain the difference between soft clustering and hard clustering in this context.

---

5. Conditional Expectation of the Observation Given the Cluster:
Show that
$$
\mathbb E[X_i\mid C_i=k]=\mu_k.
$$
Explain why $\mu_k$ can be interpreted as the center of cluster $k$.
Then compare the two conditional expectations
$$
\mathbb E[Z_i\mid X_i=x_i]
$$
and
$$
\mathbb E[X_i\mid C_i=k].
$$
Explain why the first gives the soft cluster membership of an observed point, while the second gives the mean location of a cluster.

---

6. The Complete-Data Likelihood
If the latent labels $z_i$ were known, the complete-data likelihood would be
$$
p(x_1,\dots,x_n,z_1,\dots,z_n)=\prod_{i=1}^n
\prod_{k=1}^K
\left[
\phi_k
\mathscr N(x_i\mid \mu_k,\Sigma_k)
\right]^{z_{ik}}.
$$
Take the logarithm and show that the complete-data log-likelihood is
$$
\ell_c=\sum_{i=1}^n
\sum_{k=1}^K
z_{ik}
\left[
\log \phi_k
+
\log \mathscr N(x_i\mid \mu_k,\Sigma_k)
\right].
$$
Explain why this expression would be easy to maximize if the values of $z_{ik}$ were known.

---

7. The EM Interpretation:
In practice, the latent variables $Z_i$ are not observed. The EM algorithm replaces the unknown indicators $z_{ik}$ by their conditional expectations given the observed data and current parameter estimates:
$$
z_{ik}
\quad\leadsto\quad
\mathbb E[Z_{ik}\mid X_i=x_i].
$$
That is,
$$
z_{ik}
\quad\leadsto\quad
\gamma_{ik}.
$$
This is the E-step of the EM algorithm.
Using this idea, write the expected complete-data log-likelihood:
$$
Q=\sum_{i=1}^n
\sum_{k=1}^K
\gamma_{ik}
\left[
\log \phi_k
+
\log \mathscr N(x_i\mid \mu_k,\Sigma_k)
\right].
$$
Explain why the E-step can be interpreted as a conditional update of cluster membership probabilities.

---

8. Parameter Updates:
By maximizing $Q$ with respect to the model parameters, derive the standard GMM updates:
$$
N_k=\sum_{i=1}^n \gamma_{ik},
$$
$$
\phi_k^{\text{new}}=\frac{N_k}{n},
$$
$$
\mu_k^{\text{new}}=\frac{1}{N_k}
\sum_{i=1}^n
\gamma_{ik}x_i,
$$
and
$$
\Sigma_k^{\text{new}}=\frac{1}{N_k}
\sum_{i=1}^n
\gamma_{ik}
(x_i-\mu_k^{\text{new}})
(x_i-\mu_k^{\text{new}})^T.
$$
Explain how the responsibility $\gamma_{ik}$ acts as a fractional membership weight of observation $x_i$ in cluster $k$.

---

9. Interpretation:
Write a short paragraph explaining why GMM clustering can be viewed as a repeated process of conditional updating.
Your answer should mention the following points:

* The mixture weight $\phi_k$ is the prior probability of cluster $k$.
* The Gaussian density $\mathscr N(x_i\mid \mu_k,\Sigma_k)$ measures how compatible $x_i$ is with cluster $k$.
* The responsibility $\gamma_{ik}$ is the posterior probability of cluster $k$ after observing $x_i$.
* The soft assignment vector is
$$
\mathbb E[Z_i\mid X_i=x_i].
$$

* The M-step updates the cluster parameters using these posterior membership probabilities as weights.
Conclude that Gaussian mixture clustering is probabilistic clustering based on conditional expectations of latent cluster membership variables.

---

Here is a perfectly tailored question that you can add as the final part (**Part 10**) of your assignment notebook to bridge your theoretical derivations with your code implementation:

---

10. Computational Simulation and Out-of-Sample Validation

Using the theoretical framework established in the previous parts, write a Python class named `GMMFinancialSegmenter` that implements a two-dimensional Gaussian Mixture Model (GMM) using `scikit-learn` and visualizes the results interactively using `Plotly`. Your implementation should fulfill the following criteria:

* **Data Splitting and Scaling:** Accept a dataset containing two continuous features (e.g., mimicking financial behaviors like `PURCHASES` and `CREDIT_LIMIT`), standardize the features to handle variance scaling, and split the data into an 80% training set and a 20% validation/test set.
* **EM Execution:** Fit a GMM with $K=3$ components on the training data using the Expectation-Maximization (EM) algorithm, printing whether the model successfully converged and the number of iterations required.
* **Out-of-Sample Performance:** Compute and output the average log-likelihood score over the unseen test set to validate how well the learned density functions generalize to new data.
* **Interactive Visualizations:** Implement methods to generate three distinct Plotly figures:
1. An empirical **2D Density Heatmap** of the raw training data with marginal distributions to inspect its underlying multimodal structure.
2. A **Training Assignment Plot** that overlays the training data points on top of a continuous contour map showing the maximum posterior responsibilities ($\gamma_{ik}$) computed across a fine coordinate grid.
3. A **Test Assignment Plot** that replicates the contour boundary visualization but overlays out-of-sample test data points to expose the physical regions of cluster ambiguity.



Briefly evaluate the resulting plots. Explain how the continuous background contour map visually demonstrates the soft assignment expectation vector $\mathbb{E}[Z_i \mid X_i = x_{\text{grid}}]$ that you proved analytically in Part 3.

Use the dataset

https://www.kaggle.com/datasets/arjunbhasin2013/ccdata

1. Marginal Density:
By the law of total probability, $p(x_i) = \sum_{k=1}^K P(X_i=x_i \mid C_i=k) P(C_i=k)$. Substituting the given distributions, $p(x_i) = \sum_{k=1}^K \phi_k \mathscr{N}(x_i \mid \mu_k, \Sigma_k)$. This is a Gaussian mixture density because it models the overall data distribution as a weighted convex combination ("mixture") of individual Gaussian components.  

2. Posterior Cluster Probability (Responsibility):
Applying Bayes' theorem to the given priors and likelihoods yields $\gamma_{ik}$. This is interpreted as a posterior probability because it represents our updated belief about which cluster generated the point $x_i$ after observing its specific spatial features.

3. One-Hot Encoding:
By definition of expectations for indicator variables, $\mathbb{E}[Z_{ik} \mid X_i=x_i] = 1 \cdot P(Z_{ik}=1 \mid X_i=x_i) + 0 \cdot P(Z_{ik}=0 \mid X_i=x_i) = P(C_i=k \mid X_i=x_i) = \gamma_{ik}$. Stacking these into a vector provides the soft cluster assignment, confirming it is exactly the conditional expectation $\mathbb{E}[Z_i \mid X_i=x_i]$.
4. Soft vs Hard Clustering:
Soft clustering assigns a probability vector to each point (e.g., 70% cluster A, 30% cluster B), capturing uncertainty. Hard clustering forces a mutually exclusive assignment by taking the argmax of the soft probabilities, collapsing the uncertainty into a single deterministic label.

5. Conditional Expectation given Cluster:
For a Gaussian distribution, the expected value is its mean parameter, thus $\mathbb{E}[X_i \mid C_i=k] = \mu_k$, which defines the spatial center of the cluster. $\mathbb{E}[Z_i \mid X_i=x_i]$ operates in the latent discrete space (membership probabilities), while $\mathbb{E}[X_i \mid C_i=k]$ operates in the continuous feature space (spatial locations).

6. Complete-Data Log-Likelihood:
Taking the logarithm of the given complete-data likelihood converts the product into sums:
  $$\ell_c = \sum_{i=1}^n \sum_{k=1}^K z_{ik} \left[ \log \phi_k + \log \mathscr{N}(x_i \mid \mu_k, \Sigma_k) \right]$$If $z_{ik}$ were known, this simplifies into completely decoupled sums for each cluster, allowing the parameters $(\phi_k, \mu_k, \Sigma_k)$ to be maximized independently using standard closed-form MLE formulas.

7.  Expected Complete-Data Log-Likelihood (E-step):
Substituting $z_{ik}$ with $\gamma_{ik}$ computes the expected complete-data log-likelihood ($Q$). This E-step is a conditional update because it calculates the current best guess of membership probabilities based conditionally on the current spatial parameter estimates.

8. Parameter Updates (M-step):
Maximizing $Q$ yields formulas where $\gamma_{ik}$ acts as a fractional weight. Instead of a point $x_i$ fully belonging to a single cluster, it contributes a fraction ($\gamma_{ik}$) of its value to cluster $k$'s mean and covariance calculations.

9. Interpretation Paragraph:
Gaussian Mixture Model clustering is fundamentally a repeated process of conditional updating. The mixture weight $\phi_k$ acts as the initial prior probability of a point belonging to cluster $k$. We then measure how compatible a specific observation $x_i$ is with cluster $k$ using the Gaussian density $\mathscr{N}(x_i \mid \mu_k, \Sigma_k)$. Combining these via Bayes' rule gives the responsibility $\gamma_{ik}$, representing the updated posterior probability of cluster $k$ after observation. These responsibilities form the soft assignment vector $\mathbb{E}[Z_i \mid X_i=x_i]$. Finally, the M-step performs a conditional update on the cluster spatial parameters using these posterior probabilities as fractional weights. Consequently, GMM is a fully probabilistic clustering paradigm built natively upon conditional expectations of latent variables

Implementation: Part 10 (Computational Simulation)

In [5]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

class GMMFinancialSegmenter:
    def __init__(self, n_components=3):
        self.gmm = GaussianMixture(n_components=n_components, covariance_type='full', random_state=42)
        self.scaler = StandardScaler()

    def prepare_data(self, df, features):
        X = df[features].dropna().values
        X_scaled = self.scaler.fit_transform(X)
        self.X_train, self.X_test = train_test_split(X_scaled, test_size=0.2, random_state=42)
        return self.X_train, self.X_test

    def fit_and_evaluate(self):
        self.gmm.fit(self.X_train)
        print(f"Model Converged: {self.gmm.converged_}")
        print(f"Iterations: {self.gmm.n_iter_}")

        test_score = self.gmm.score(self.X_test)
        print(f"Out-of-Sample Average Log-Likelihood: {test_score:.4f}")

    def plot_heatmap(self):
        fig = px.density_heatmap(x=self.X_train[:, 0], y=self.X_train[:, 1],
                                 marginal_x="histogram", marginal_y="histogram",
                                 title="2D Density Heatmap of Training Data")
        fig.show()

    def plot_assignments(self, data, title):
        # Create a mesh grid
        x_min, x_max = data[:, 0].min() - 1, data[:, 0].max() + 1
        y_min, y_max = data[:, 1].min() - 1, data[:, 1].max() + 1
        xx, yy = np.meshgrid(np.linspace(x_min, x_max, 100), np.linspace(y_min, y_max, 100))

        # Predict responsibilities on grid (Hard assignment for background)
        Z = self.gmm.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

        fig = go.Figure()
        fig.add_trace(go.Contour(x=np.linspace(x_min, x_max, 100),
                                 y=np.linspace(y_min, y_max, 100),
                                 z=Z, colorscale='Viridis', opacity=0.3, showscale=False))

        preds = self.gmm.predict(data)
        fig.add_trace(go.Scatter(x=data[:, 0], y=data[:, 1], mode='markers',
                                 marker=dict(color=preds, colorscale='Viridis', line_width=1)))
        fig.update_layout(title=title, xaxis_title="Feature 1 (Scaled)", yaxis_title="Feature 2 (Scaled)")
        fig.show()

# --- Execution Example ---
# NOTE: Replace with your actual downloaded Kaggle dataset path
# df = pd.read_csv('CC GENERAL.csv')
# df['CREDIT_LIMIT'] = df['CREDIT_LIMIT'].fillna(df['CREDIT_LIMIT'].median())

# Mock data to demonstrate class functionality natively in Colab
np.random.seed(42)
mock_data = pd.DataFrame({
    'PURCHASES': np.concatenate([np.random.normal(500, 200, 300), np.random.normal(3000, 500, 300)]),
    'CREDIT_LIMIT': np.concatenate([np.random.normal(1500, 400, 300), np.random.normal(6000, 1000, 300)])
})

segmenter = GMMFinancialSegmenter(n_components=3)
X_train, X_test = segmenter.prepare_data(mock_data, ['PURCHASES', 'CREDIT_LIMIT'])
segmenter.fit_and_evaluate()
segmenter.plot_heatmap()
segmenter.plot_assignments(X_train, "Training Assignment Plot")
segmenter.plot_assignments(X_test, "Test Assignment Plot")

Model Converged: True
Iterations: 5
Out-of-Sample Average Log-Likelihood: -0.7082
